In [14]:
import hydra
from langchain_core.messages import HumanMessage, SystemMessage

from social_groups.trialrunner.config import get_llm
from social_groups.trialrunner.utils.cli_utils import setup_config

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
USED_CASE = 2

cases = [
    [
        "+experiment=final/baseline_ministral3",
        "experiment/hetero/backend@experiment.strategy.configuration.backend=ministral3-3b",
    ],
    ["+experiment=final/baseline"],
    ["+experiment=final/baseline_qwen35"],
]

In [16]:
with hydra.initialize(version_base=None, config_path="../configs/trials"):
    cfg = hydra.compose(
        config_name="local",
        overrides=cases[USED_CASE],
    )

config = setup_config(cfg)

In [17]:
llm = get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
)

llm

ChatQwen(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x12586b350>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1262d79d0>, root_client=<openai.OpenAI object at 0x125b3d610>, root_async_client=<openai.AsyncOpenAI object at 0x125861f50>, model_name='Qwen/Qwen3.5-4B', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='http://127.0.0.1:40504/v1', seed=42, top_p=0.1, max_tokens=4096, api_key=SecretStr('**********'), api_base='http://127.0.0.1:40504/v1')

In [18]:
messages = [
    SystemMessage("You are a professor. Answer the following:"),
    HumanMessage("This is a test. What os 3 + 3? (A) 6 (B) 12 (C) 15"),
    # HumanMessage(
    #     "You have to first think about the correct answer. Then submit your answer using the given tool."
    # ),
]

### With Tool Call

In [19]:
from social_groups.trialrunner.utils.tool_calls import parse_tool_call_arguments
from langchain_core.tools import tool


@tool(return_direct=True)
def propose_solution(correct_answer: str, reasoning: str):
    """
    Submit the answer to the question that you think is correct.
    Please provide extensive reasoning on why this is correct,
    which assumptions and knowledge was used to retrieve the answer
    and what argumentation is needed to come to the conclusion.

    :arg correct_answer: The correct answer-letter of the question. in the form: (X)
    :arg reasoning: Fully specified reasoning path to retrieve this answer.
    """
    pass


answer = llm.bind_tools([propose_solution]).invoke(
    messages
)

print(answer)
print()
print(parse_tool_call_arguments(answer)["correct_answer"])

content="The user is asking a simple math question: 3 + 3 = ?\n\nThis is a basic arithmetic question that doesn't require any special tools. 3 + 3 = 6.\n\nLooking at the options:\n(A) 6\n(B) 12\n(C) 15\n\nThe correct answer is (A) 6.\n\nI should provide a clear, straightforward answer with reasoning.\n</think>\n\n" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 244, 'prompt_tokens': 401, 'total_tokens': 645, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'dashscope', 'model_name': 'Qwen/Qwen3.5-4B', 'system_fingerprint': None, 'id': 'chatcmpl-bcc2bf79d57ecbd1', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d518a-7d51-7fc0-84ce-d6cf86beab93-0' tool_calls=[{'name': 'propose_solution', 'args': {'correct_answer': '(A)', 'reasoning': "This is a basic arithmetic question asking for the sum of 3 and 3. When we add 3 + 3, we get 6. This is a fundamental addition operation that doesn't require a

### Standard call

In [20]:
llm.invoke(messages)

KeyboardInterrupt: 

### With Thinking

In [ ]:
get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
    with_thinking=True,
).invoke(messages)

### Without Thinking

In [ ]:
get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
    with_thinking=False,
).invoke(messages)

### With Structured Output

In [22]:
from pydantic import Field
from pydantic import BaseModel


class AnswerResponseFormat(BaseModel):
    response: str = Field(
        ..., description="The answer to the question in the form 'The answer is (X)'."
    )


get_llm(
    config=config.experiment.strategy.configuration.llm,
    backend=config.experiment.strategy.configuration.backend,
    with_thinking=False,
).with_structured_output(
    AnswerResponseFormat,
    strict=True,
    include_raw=True,
    method="json_schema" if "mistral" in config.experiment.strategy.configuration.backend.model_name else "function_calling",
).invoke(messages)

{'raw': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 323, 'total_tokens': 342, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'dashscope', 'model_name': 'Qwen/Qwen3.5-4B', 'system_fingerprint': None, 'id': 'chatcmpl-a5aebfc1edcf92c1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d518a-c3b5-7e91-b71d-845d68a924f6-0', tool_calls=[{'name': 'AnswerResponseFormat', 'args': {'response': 'The answer is (A) 6'}, 'id': 'chatcmpl-tool-a5770b81581b2a66', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 323, 'output_tokens': 19, 'total_tokens': 342, 'input_token_details': {}, 'output_token_details': {}}),
 'parsed': AnswerResponseFormat(response='The answer is (A) 6'),
 'parsing_error': None}